# Python Threading
Covers: Thread class, daemon threads, Lock/RLock/Semaphore/Event/Condition, ThreadPoolExecutor, GIL, thread-safe data structures

## 1. Thread Class — Basic Usage

In [ ]:
import threading
import time

# Method 1: Pass a function
def worker(name, delay):
    print(f'Thread {name} starting')
    time.sleep(delay)
    print(f'Thread {name} done after {delay}s')

print('Starting threads...')
t1 = threading.Thread(target=worker, args=('A', 0.3))
t2 = threading.Thread(target=worker, args=('B', 0.1))
t3 = threading.Thread(target=worker, args=('C', 0.2))

start = time.perf_counter()
t1.start(); t2.start(); t3.start()
t1.join(); t2.join(); t3.join()
elapsed = time.perf_counter() - start
print(f'All done in {elapsed:.2f}s (sequential would be 0.6s)')

# Method 2: Subclass Thread
class ComputeThread(threading.Thread):
    def __init__(self, data):
        super().__init__()
        self.data = data
        self.result = None
    
    def run(self):
        self.result = sum(x**2 for x in self.data)

t = ComputeThread(range(1000))
t.start()
t.join()
print(f'\nComputed sum of squares: {t.result}')

## 2. Lock — Preventing Race Conditions

In [ ]:
import threading

# Demonstrate race condition
counter_unsafe = 0
counter_safe = 0
lock = threading.Lock()

def increment_unsafe(n):
    global counter_unsafe
    for _ in range(n):
        counter_unsafe += 1  # NOT atomic! read-modify-write

def increment_safe(n):
    global counter_safe
    for _ in range(n):
        with lock:  # atomic read-modify-write
            counter_safe += 1

N = 10_000
NUM_THREADS = 5

# Unsafe
threads = [threading.Thread(target=increment_unsafe, args=(N,)) for _ in range(NUM_THREADS)]
for t in threads: t.start()
for t in threads: t.join()
print(f'Unsafe counter: {counter_unsafe} (expected {N * NUM_THREADS})')
print(f'  Lost updates: {N * NUM_THREADS - counter_unsafe}')

# Safe
threads = [threading.Thread(target=increment_safe, args=(N,)) for _ in range(NUM_THREADS)]
for t in threads: t.start()
for t in threads: t.join()
print(f'\nSafe counter: {counter_safe} (expected {N * NUM_THREADS})')
print(f'  Correct: {counter_safe == N * NUM_THREADS}')

## 3. RLock, Semaphore, Event

In [ ]:
import threading
import time

# RLock — reentrant lock
rlock = threading.RLock()

def recursive_sum(n, depth=0):
    with rlock:  # same thread can acquire multiple times
        if n <= 0:
            return 0
        return n + recursive_sum(n - 1, depth + 1)

print('RLock recursive sum(5):', recursive_sum(5))

# Semaphore — limit concurrent access
MAX_CONCURRENT = 3
semaphore = threading.Semaphore(MAX_CONCURRENT)
active_count = 0
max_active = 0
count_lock = threading.Lock()

def limited_task(task_id):
    global active_count, max_active
    with semaphore:
        with count_lock:
            active_count += 1
            max_active = max(max_active, active_count)
        time.sleep(0.05)
        with count_lock:
            active_count -= 1

threads = [threading.Thread(target=limited_task, args=(i,)) for i in range(10)]
for t in threads: t.start()
for t in threads: t.join()
print(f'\nSemaphore: max concurrent = {max_active} (limit was {MAX_CONCURRENT})')

# Event — signal between threads
ready_event = threading.Event()
results = []

def worker_waiting(worker_id):
    ready_event.wait()  # block until event is set
    results.append(f'Worker {worker_id} ran')

workers = [threading.Thread(target=worker_waiting, args=(i,)) for i in range(4)]
for w in workers: w.start()

time.sleep(0.1)
print(f'\nBefore event.set(): {len(results)} workers ran')
ready_event.set()  # unblock all workers at once
for w in workers: w.join()
print(f'After event.set(): {len(results)} workers ran')

## 4. ThreadPoolExecutor

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
import random

def simulate_io_task(task_id):
    """Simulate an I/O-bound task (network request, file read, etc.)"""
    delay = random.uniform(0.05, 0.2)
    time.sleep(delay)
    if task_id == 7:  # simulate occasional failure
        raise ValueError(f'Task {task_id} failed')
    return f'Task {task_id} completed in {delay:.3f}s'

task_ids = list(range(12))

# executor.map — simple, ordered results
print('=== executor.map (ordered) ===')
start = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as executor:
    try:
        results = list(executor.map(simulate_io_task, task_ids))
    except ValueError as e:
        print(f'Error: {e}')
elapsed = time.perf_counter() - start
print(f'Completed in {elapsed:.2f}s')

# executor.submit + as_completed — unordered, handles exceptions per-task
print('\n=== as_completed (unordered, per-task error handling) ===')
start = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {executor.submit(simulate_io_task, tid): tid for tid in task_ids}
    
    completed = 0
    errors = 0
    for future in as_completed(futures):
        task_id = futures[future]
        try:
            result = future.result()
            completed += 1
        except ValueError as e:
            errors += 1
            print(f'  Error on task {task_id}: {e}')

elapsed = time.perf_counter() - start
print(f'Completed: {completed}, Errors: {errors}, Time: {elapsed:.2f}s')

## 5. GIL — CPU-bound vs I/O-bound

In [ ]:
import threading
import time

def cpu_bound(n):
    """CPU-intensive work — GIL prevents true parallelism."""
    return sum(i * i for i in range(n))

def io_bound(delay):
    """I/O work — GIL is released, threads run concurrently."""
    time.sleep(delay)  # GIL released during sleep
    return delay

N = 2_000_000
DELAY = 0.3

# CPU-bound: threading doesn't help (GIL)
print('=== CPU-bound ===')
start = time.perf_counter()
cpu_bound(N); cpu_bound(N)
sequential_cpu = time.perf_counter() - start

start = time.perf_counter()
t1 = threading.Thread(target=cpu_bound, args=(N,))
t2 = threading.Thread(target=cpu_bound, args=(N,))
t1.start(); t2.start()
t1.join(); t2.join()
threaded_cpu = time.perf_counter() - start

print(f'Sequential: {sequential_cpu:.3f}s')
print(f'Threaded:   {threaded_cpu:.3f}s')
print(f'Speedup: {sequential_cpu/threaded_cpu:.2f}x (expected ~1.0 due to GIL)')

# I/O-bound: threading helps!
print('\n=== I/O-bound ===')
start = time.perf_counter()
io_bound(DELAY); io_bound(DELAY)
sequential_io = time.perf_counter() - start

start = time.perf_counter()
t1 = threading.Thread(target=io_bound, args=(DELAY,))
t2 = threading.Thread(target=io_bound, args=(DELAY,))
t1.start(); t2.start()
t1.join(); t2.join()
threaded_io = time.perf_counter() - start

print(f'Sequential: {sequential_io:.3f}s')
print(f'Threaded:   {threaded_io:.3f}s')
print(f'Speedup: {sequential_io/threaded_io:.2f}x (expected ~2.0)')
print('\nConclusion: Use threading for I/O-bound, multiprocessing for CPU-bound')

## 6. Thread-Safe Queue — Producer/Consumer

In [ ]:
import threading
import queue
import time
import random

# Producer-Consumer pattern with queue.Queue
work_queue = queue.Queue(maxsize=5)
results = []
results_lock = threading.Lock()

def producer(items):
    for item in items:
        work_queue.put(item)  # blocks if queue is full
        print(f'  Produced: {item} (queue size: {work_queue.qsize()})')
        time.sleep(0.02)
    # Send sentinel values to stop consumers
    for _ in range(2):  # one per consumer
        work_queue.put(None)

def consumer(name):
    while True:
        item = work_queue.get()  # blocks if queue is empty
        if item is None:
            work_queue.task_done()
            break
        # Process item
        result = item ** 2
        with results_lock:
            results.append(result)
        work_queue.task_done()

items = list(range(1, 11))
prod = threading.Thread(target=producer, args=(items,))
cons1 = threading.Thread(target=consumer, args=('Consumer-1',))
cons2 = threading.Thread(target=consumer, args=('Consumer-2',))

prod.start(); cons1.start(); cons2.start()
prod.join(); cons1.join(); cons2.join()

print(f'\nProcessed {len(results)} items')
print(f'Results (sorted): {sorted(results)}')
print(f'Expected: {[x**2 for x in items]}')